# 1. Fixed-fusion uncertainty attribution across repeated seeds

This notebook repeats the earlier participant-level uncertainty-attribution analysis on the **15 completed fixed-equal-fusion checkpoints**: five held-out folds for each of seeds 17, 42 and 73.

It performs no training. For every seed/fold checkpoint it reconstructs the exact saved model and test partition, reproduces the previously saved test predictions, and only then accepts the attribution outputs.

The analysis measures:

- uncertainty and evidence from every available modality opinion;
- pairwise conflict between modality opinions;
- sequential modality-specific evidence fusion behaviour;
- changes in evidence pathway and Hybrid predictions when one available modality opinion is removed;
- stability of modality rankings across the three complete out-of-fold seed runs.

## 1.1. How to run this notebook safely

Select an **A100 GPU** runtime and choose **Run all**. Each completed seed/fold analysis is written atomically to its own output directory and receives a `_SUCCESS.json` file only after all tables and prediction-reproduction checks pass.

If Colab disconnects, reconnect and run the notebook from the beginning. Verified completed seed/fold analyses will be loaded and skipped. Only the interrupted or invalid run will be recomputed.

The five folds form one complete 544-participant out-of-fold evaluation for each seed. Therefore, the main replication count is **three seeds**, not fifteen independent models.

In [ ]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import gc
import hashlib
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass


PROJECT_ROOT = Path("/content/drive/MyDrive/adni_mri")
MODEL_ROOT = PROJECT_ROOT / "models" / "3mt_tmc_evidential"

EXPERIMENT_NAME = "gated_cmt_fixed_equal_fusion_md050_seed_sensitivity"
TASK_NAME = "mci_prognosis"
SEEDS = (17, 42, 73)
FOLDS = (0, 1, 2, 3, 4)
EXPECTED_PARTICIPANTS_PER_SEED = 544

INPUT_ROOT = MODEL_ROOT / "final_task_ready_inputs" / TASK_NAME
SEED_TASK_ROOT = MODEL_ROOT / "experiments" / EXPERIMENT_NAME / TASK_NAME

OUTPUT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
    / "repeated_seed_uncertainty_attribution"
    / TASK_NAME
)
PER_RUN_ROOT = OUTPUT_ROOT / "per_run"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"

for directory in (OUTPUT_ROOT, PER_RUN_ROOT, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 2
NUM_WORKERS = 0
CLASSIFICATION_THRESHOLD = 0.50
NUMERICAL_EPSILON = 1e-8
REPRODUCTION_ATOL = 2e-5

MODALITY_DISPLAY_NAMES = {
    "demographics": "Demographics",
    "cognitive_functional": "Cognitive/Functional",
    "csf": "CSF",
    "plasma": "Plasma",
    "apoe": "APOE",
    "mri": "MRI",
}

RUN_TABLE_FILENAMES = {
    "participant": "participant_uncertainty_attribution.csv",
    "modality": "modality_evidential_outputs_long.csv",
    "pairwise": "pairwise_modality_conflict_long.csv",
    "fusion": "sequential_tmc_fusion_history_long.csv",
    "loo": "leave_one_modality_out_attribution_long.csv",
}

print("=" * 88)
print("FIXED-FUSION UNCERTAINTY ATTRIBUTION ACROSS THREE COMPLETE SEED RUNS")
print("=" * 88)
print(f"Existing checkpoints: {SEED_TASK_ROOT}")
print(f"New attribution outputs: {OUTPUT_ROOT}")
print(f"Seeds: {SEEDS}; folds per seed: {FOLDS}")
print("This notebook performs inference only. It does not train or alter a checkpoint.")

## 1.2. Exact prepared-input schema and modality contract

In [ ]:
final_model_schema = {
    "identifier_columns": [
        "RID",
        "PTID",
    ],

    "audit_label_columns": [
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ],

    "model_target_column":
        "MODEL_TARGET",

    "split_columns": [
        "OUTER_FOLD",
        "DATA_ROLE",
    ],

    "scaled_continuous_columns": [
        "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
        "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        "ADAS__TOTSCORE__Z",
        "ADAS__TOTAL13__Z",
        "MMSE__MMSE_TOTAL_SCORE__Z",
        "MMSE__MMSE_ORIENTATION_SCORE__Z",
        "MMSE__MMSE_REGISTRATION_SCORE__Z",
        "MMSE__MMSE_ATTENTION_SCORE__Z",
        "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
        "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
        "FAQ__FAQTOTAL__Z",
        "CSF__ABETA40__Z",
        "CSF__ABETA42__Z",
        "CSF__TAU__Z",
        "CSF__PTAU__Z",
        "CSF__ABETA42_40_RATIO__Z",
        "PLASMA__pT217_F__Z",
        "PLASMA__AB42_F__Z",
        "PLASMA__AB40_F__Z",
        "PLASMA__AB42_AB40_F__Z",
        "PLASMA__pT217_AB42_F__Z",
        "PLASMA__NfL_Q__Z",
        "PLASMA__GFAP_Q__Z",
        "PLASMA__NfL_F__Z",
        "PLASMA__GFAP_F__Z",
    ],

    "encoded_categorical_columns": [
        "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
        "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        "APOE__APOE4_ALLELE_COUNT__IDX",
    ],

    "mri_path_columns": [
        "MRI__NORMALIZED_T1_NPY_PATH",
    ],

    "branch_mask_columns": [
        "BRANCH_MASK__DEMOGRAPHICS",
        "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
        "BRANCH_MASK__CSF",
        "BRANCH_MASK__PLASMA",
        "BRANCH_MASK__APOE",
        "BRANCH_MASK__MRI",
    ],

    "feature_mask_columns": [
        "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
        "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        "FEATURE_MASK__ADAS_TOTSCORE",
        "FEATURE_MASK__ADAS_TOTAL13",
        "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
        "FEATURE_MASK__FAQ_FAQTOTAL",
        "FEATURE_MASK__CSF_ABETA40",
        "FEATURE_MASK__CSF_ABETA42",
        "FEATURE_MASK__CSF_TAU",
        "FEATURE_MASK__CSF_PTAU",
        "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        "FEATURE_MASK__PLASMA_pT217_F",
        "FEATURE_MASK__PLASMA_AB42_F",
        "FEATURE_MASK__PLASMA_AB40_F",
        "FEATURE_MASK__PLASMA_AB42_AB40_F",
        "FEATURE_MASK__PLASMA_pT217_AB42_F",
        "FEATURE_MASK__PLASMA_NfL_Q",
        "FEATURE_MASK__PLASMA_GFAP_Q",
        "FEATURE_MASK__PLASMA_NfL_F",
        "FEATURE_MASK__PLASMA_GFAP_F",
        "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
    ],
}

identifier_columns = final_model_schema["identifier_columns"]
target_column = final_model_schema["model_target_column"]
branch_mask_columns = final_model_schema["branch_mask_columns"]
feature_mask_columns = final_model_schema["feature_mask_columns"]

# ============================================================
# 3. Defining the multimodal dataset-output contract
# ============================================================

# ------------------------------------------------------------
# Branch-specific predictor columns
# ------------------------------------------------------------

# I organise the prepared continuous and categorical columns into
# the six modality branches used by the architecture.

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}

print("Branch-mask order:")
for index, column in enumerate(branch_mask_columns):
    print(f"{index}: {column}")

## 1.3. Exact multimodal dataset used by the trained checkpoints

In [ ]:
# ============================================================
# 4. Implementing the multimodal PyTorch dataset
# ============================================================

import numpy as np
import torch

from torch.utils.data import Dataset


# ------------------------------------------------------------
# Expected prepared MRI shape
# ------------------------------------------------------------

# I preserve the spatial dimensions produced by the completed
# MRI preprocessing pipeline.
MRI_SPATIAL_SHAPE = (177, 213, 183)

# I add one channel dimension when returning an MRI tensor.
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)


# ------------------------------------------------------------
# Multimodal PyTorch dataset
# ------------------------------------------------------------

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # I reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # I retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # I preserve the authoritative mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri


    def __len__(self):
        return len(self.dataframe)


    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # I load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # I ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)


    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # --------------------------------------------------------
        # Demographics
        # --------------------------------------------------------

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Cognitive and functional measures
        # --------------------------------------------------------

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # CSF
        # --------------------------------------------------------

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Plasma
        # --------------------------------------------------------

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # APOE
        # --------------------------------------------------------

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # MRI
        # --------------------------------------------------------

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Complete sample
        # --------------------------------------------------------

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample

## 1.4. Exact fixed-equal-fusion architecture

The following definitions are copied from the original fixed-fusion attribution notebook. Strict state-dictionary loading and participant-level prediction reproduction below protect against silent architectural mismatch.

In [ ]:
# ============================================================
# 7. Building the modality-specific encoders
# ============================================================

import math

import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Shared representation dimension
# ------------------------------------------------------------

# I project every modality into one common latent space so that
# all branches can later enter the same cross-modal transformers.
MODALITY_EMBEDDING_DIM = 64


# ------------------------------------------------------------
# Reusable scalar encoder
# ------------------------------------------------------------

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        values,
        feature_mask,
    ):
        # I supply both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Demographics encoder
# ------------------------------------------------------------

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# APOE encoder
# ------------------------------------------------------------

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Residual 3D downsampling block
# ------------------------------------------------------------

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original interaction pathway image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()


    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )


# ------------------------------------------------------------
# interaction pathway-style CNN-transformer MRI encoder
# ------------------------------------------------------------

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # --------------------------------------------------------
        # Initial convolutional stem
        # --------------------------------------------------------

        # I use two initial 3D convolutions, following the broad
        # structure shown in the interaction pathway image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original interaction pathway inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )


        # --------------------------------------------------------
        # Four residual downsampling blocks
        # --------------------------------------------------------

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )


        # --------------------------------------------------------
        # Determine the resulting patch grid
        # --------------------------------------------------------

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )


        # --------------------------------------------------------
        # Learned positional embeddings
        # --------------------------------------------------------

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )


        # --------------------------------------------------------
        # Patch-wise transformer encoder
        # --------------------------------------------------------

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )


        # --------------------------------------------------------
        # Projection to the shared modality dimension
        # --------------------------------------------------------

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )


    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # I flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # I add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )


# ------------------------------------------------------------
# Complete set of six modality encoders
# ------------------------------------------------------------

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )


    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations

In [ ]:
# ============================================================
# 8. Applying branch masks to the encoded modalities
# ============================================================

# ------------------------------------------------------------
# Fixed modality order
# ------------------------------------------------------------

# I use one explicit modality order throughout the architecture.
# This order matches the prepared branch-mask columns.
MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]


# ------------------------------------------------------------
# Mask-aware encoder wrapper
# ------------------------------------------------------------

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )


    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # I add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask


    def forward(self, modalities):
        # I first obtain the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # I then suppress every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # I return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }

In [ ]:
# ============================================================
# 9. Building the availability-gated interaction pathway cascade
# ============================================================

# ------------------------------------------------------------
# Fixed cascade order
# ------------------------------------------------------------

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]


# ------------------------------------------------------------
# One Cascaded Modality Transformer
# ------------------------------------------------------------

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }


# ------------------------------------------------------------
# Complete six-stage availability-gated cascade
# ------------------------------------------------------------

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )


    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }

In [ ]:
# ============================================================
# 10. Producing independent modality-specific evidential opinions
# ============================================================

# ------------------------------------------------------------
# Binary prognosis class definition
# ------------------------------------------------------------

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]


# ------------------------------------------------------------
# One modality-specific evidential head
# ------------------------------------------------------------

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(
        self,
        representation,
        branch_mask,
    ):
        # I use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "belief": effective_evidence / strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }


# ------------------------------------------------------------
# Independent evidential heads for all six modalities
# ------------------------------------------------------------

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )


    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions

In [ ]:
# ============================================================
# 11. Fusing the independent modality opinions with evidence pathway
# ============================================================

# ------------------------------------------------------------
# Reduced Dempster-Shafer combination rule
# ------------------------------------------------------------

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )


    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }


    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]


        # --------------------------------------------------------
        # Conflict mass
        # --------------------------------------------------------

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)


        # --------------------------------------------------------
        # Fused belief and uncertainty masses
        # --------------------------------------------------------

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation


        # --------------------------------------------------------
        # Recover the fused Dirichlet opinion
        # --------------------------------------------------------

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }


    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # I retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }


        # --------------------------------------------------------
        # Final fused opinion
        # --------------------------------------------------------

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }

In [ ]:
# ============================================================
# 12. Producing the interaction-aware interaction pathway evidential opinion
# ============================================================

# ------------------------------------------------------------
# Auxiliary classifier for one intermediate interaction pathway query
# ------------------------------------------------------------

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(self, representation):
        return self.network(
            representation
        )


# ------------------------------------------------------------
# Joint interaction pathway evidential and auxiliary heads
# ------------------------------------------------------------

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )


    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # --------------------------------------------------------
        # Intermediate auxiliary predictions
        # --------------------------------------------------------

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )


        # --------------------------------------------------------
        # Final joint evidential opinion
        # --------------------------------------------------------

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final interaction pathway query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }

In [ ]:
# ============================================================
# 13. Combining interaction pathway and evidence pathway with fixed equal fusion
# ============================================================

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )


class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )


    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )


    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }

In [ ]:
# ============================================================
# 14. Assembling the complete end-to-end interaction-evidence model
# ============================================================

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )


        # --------------------------------------------------------
        # Modality-specific encoders
        # --------------------------------------------------------

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )


        # --------------------------------------------------------
        # Independent modality evidential pathway
        # --------------------------------------------------------

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )


        # --------------------------------------------------------
        # Final fixed 50/50 hybrid fusion
        # --------------------------------------------------------

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )


    # ------------------------------------------------------------
    # Training-time modality dropout
    # ------------------------------------------------------------

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )


        # --------------------------------------------------------
        # Sample branch-retention indicators
        # --------------------------------------------------------

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )


        # --------------------------------------------------------
        # Prevent complete information removal
        # --------------------------------------------------------

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # I randomly restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0


        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )


    # ------------------------------------------------------------
    # Construct effective modality dictionaries
    # ------------------------------------------------------------

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities


    # ------------------------------------------------------------
    # Complete forward pass
    # ------------------------------------------------------------

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # --------------------------------------------------------
        # Apply training-time modality dropout
        # --------------------------------------------------------

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )


        # --------------------------------------------------------
        # Encode all six modalities
        # --------------------------------------------------------

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. I use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]


        # --------------------------------------------------------
        # Independent modality opinions and modality-specific evidence fusion
        # --------------------------------------------------------

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]


        # --------------------------------------------------------
        # Final fixed-equal hybrid opinion
        # --------------------------------------------------------

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )


        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }

## 1.5. Checkpoint, evidential-fusion and restart-safety utilities

In [ ]:
def move_nested_to_device(value, device):
    if isinstance(value, torch.Tensor):
        return value.to(device, non_blocking=True)

    if isinstance(value, dict):
        return {
            key: move_nested_to_device(item, device)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            move_nested_to_device(item, device)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            move_nested_to_device(item, device)
            for item in value
        )

    return value


def training_run_root(seed, fold):
    return SEED_TASK_ROOT / f"seed_{seed}" / f"fold_{fold}"


def checkpoint_path_for_seed_fold(seed, fold):
    return (
        training_run_root(seed, fold)
        / "checkpoints"
        / "best_validation_auc_checkpoint.pt"
    )


def input_path_for_fold(fold):
    return (
        INPUT_ROOT
        / f"mci_prognosis_outer_fold_{fold}_final_task_ready.csv"
    )


def build_test_loader(fold):
    fold_path = input_path_for_fold(fold)

    if not fold_path.exists():
        raise FileNotFoundError(
            f"Prepared fold table was not found: {fold_path}"
        )

    fold_table = pd.read_csv(fold_path)

    test_table = (
        fold_table.loc[
            fold_table["DATA_ROLE"].astype(str).str.lower() == "test"
        ]
        .reset_index(drop=True)
    )

    if test_table.empty:
        raise ValueError(f"Fold {fold} contains no test participants.")

    dataset = ADNIMultimodalDataset(
        dataframe=test_table,
        column_contract=dataset_column_contract,
        branch_mask_order=branch_mask_columns,
        feature_mask_order=feature_mask_columns,
        target_column=target_column,
        load_mri=True,
    )

    loader = DataLoader(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

    return test_table, loader


def opinion_from_alpha(alpha):
    alpha = alpha.float()

    strength = alpha.sum(dim=-1, keepdim=True)
    evidence = alpha - 1.0
    belief = evidence / strength
    uncertainty = alpha.shape[-1] / strength
    probabilities = alpha / strength

    return {
        "alpha": alpha,
        "evidence": evidence,
        "belief": belief,
        "uncertainty": uncertainty,
        "strength": strength,
        "probabilities": probabilities,
    }


def combine_two_alphas(alpha_a, alpha_b, number_of_classes=2):
    opinion_a = opinion_from_alpha(alpha_a)
    opinion_b = opinion_from_alpha(alpha_b)

    belief_a = opinion_a["belief"]
    belief_b = opinion_b["belief"]
    uncertainty_a = opinion_a["uncertainty"]
    uncertainty_b = opinion_b["uncertainty"]

    outer = belief_a.unsqueeze(-1) * belief_b.unsqueeze(-2)
    total_product = outer.sum(dim=(-2, -1))
    same_class = torch.diagonal(
        outer,
        dim1=-2,
        dim2=-1,
    ).sum(dim=-1)

    conflict = total_product - same_class

    normalisation = (
        1.0 - conflict
    ).clamp_min(NUMERICAL_EPSILON).unsqueeze(-1)

    fused_belief = (
        belief_a * belief_b
        + belief_a * uncertainty_b
        + belief_b * uncertainty_a
    ) / normalisation

    fused_uncertainty = (
        uncertainty_a * uncertainty_b
    ) / normalisation

    fused_strength = (
        number_of_classes
        / fused_uncertainty.clamp_min(NUMERICAL_EPSILON)
    )

    fused_evidence = fused_belief * fused_strength
    fused_alpha = fused_evidence + 1.0

    result = opinion_from_alpha(fused_alpha)
    result["conflict"] = conflict.unsqueeze(-1)

    return result


def fuse_available_opinions(
    modality_alpha,
    availability,
    modality_order,
):
    available_names = [
        name
        for name in modality_order
        if int(availability[name]) == 1
    ]

    if not available_names:
        raise ValueError(
            "A participant has no available modality opinions."
        )

    first_name = available_names[0]
    fused = opinion_from_alpha(
        modality_alpha[first_name].unsqueeze(0)
    )

    stage_rows = []

    for modality_name in available_names[1:]:
        next_alpha = modality_alpha[modality_name].unsqueeze(0)

        combined = combine_two_alphas(
            fused["alpha"],
            next_alpha,
            number_of_classes=2,
        )

        stage_rows.append(
            {
                "ADDED_MODALITY": modality_name,
                "CONFLICT": float(
                    combined["conflict"].item()
                ),
                "FUSED_UNCERTAINTY": float(
                    combined["uncertainty"].item()
                ),
            }
        )

        fused = combined

    return fused, stage_rows, available_names


def pairwise_conflict(alpha_a, alpha_b):
    opinion_a = opinion_from_alpha(alpha_a.unsqueeze(0))
    opinion_b = opinion_from_alpha(alpha_b.unsqueeze(0))

    belief_a = opinion_a["belief"]
    belief_b = opinion_b["belief"]

    outer = belief_a.unsqueeze(-1) * belief_b.unsqueeze(-2)
    total_product = outer.sum(dim=(-2, -1))
    same_class = torch.diagonal(
        outer,
        dim1=-2,
        dim2=-1,
    ).sum(dim=-1)

    return float((total_product - same_class).item())


def recalculate_hybrid_from_tmc(
    joint_evidence,
    tmc_evidence,
    three_mt_scale,
    tmc_scale,
):
    final_evidence = (
        0.5 * three_mt_scale * joint_evidence
        + 0.5 * tmc_scale * tmc_evidence
    )

    final_alpha = final_evidence + 1.0
    final_strength = final_alpha.sum()

    probabilities = final_alpha / final_strength
    uncertainty = 2.0 / final_strength

    return {
        "p_pMCI": float(probabilities[1].item()),
        "uncertainty": float(uncertainty.item()),
    }


def classify_uncertainty_source(
    available_count,
    max_modality_uncertainty,
    max_pairwise_conflict,
    strongest_uncertainty_reduction,
):
    # Descriptive categories rather than causal ground truth.
    if available_count < len(MODALITY_ORDER):
        missing_flag = True
    else:
        missing_flag = False

    high_weakness = max_modality_uncertainty >= 0.50
    high_conflict = max_pairwise_conflict >= 0.10
    meaningful_loo_reduction = strongest_uncertainty_reduction >= 0.02

    if high_conflict and meaningful_loo_reduction:
        return "Cross-modality conflict"

    if high_weakness:
        return "Weak modality evidence"

    if missing_flag:
        return "Limited modality availability"

    return "Diffuse / no dominant source"


def attribution_run_root(seed, fold):
    return PER_RUN_ROOT / f"seed_{seed}" / f"fold_{fold}"


def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_csv(dataframe, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    dataframe.to_csv(temporary, index=False)
    os.replace(temporary, destination)


def atomic_write_json(payload, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True)
    os.replace(temporary, destination)


def validate_training_artifacts(seed, fold):
    root = training_run_root(seed, fold)
    paths = {
        "configuration": root / "history" / "training_configuration.json",
        "checkpoint": root / "checkpoints" / "best_validation_auc_checkpoint.pt",
        "predictions": root / "predictions" / "test_predictions.csv",
        "metrics": root / "predictions" / "test_metrics.json",
        "success": root / "_SUCCESS.json",
        "input": input_path_for_fold(fold),
    }

    missing = [name for name, path in paths.items() if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"Seed {seed}, fold {fold} is missing required artifacts: {missing}. Root: {root}"
        )

    with open(paths["configuration"], "r", encoding="utf-8") as handle:
        configuration = json.load(handle)
    with open(paths["metrics"], "r", encoding="utf-8") as handle:
        metrics = json.load(handle)
    with open(paths["success"], "r", encoding="utf-8") as handle:
        success = json.load(handle)

    checks = {
        "configuration experiment": configuration.get("experiment_name") == EXPERIMENT_NAME,
        "configuration task": configuration.get("task") == TASK_NAME,
        "configuration seed": int(configuration.get("random_seed", -1)) == int(seed),
        "configuration fold": int(configuration.get("fold", -1)) == int(fold),
        "metrics experiment": metrics.get("experiment_name") == EXPERIMENT_NAME,
        "metrics task": metrics.get("task") == TASK_NAME,
        "metrics seed": int(metrics.get("random_seed", -1)) == int(seed),
        "metrics fold": int(metrics.get("fold", -1)) == int(fold),
        "fixed fusion": metrics.get("fusion_rule") == "fixed_equal_0.5_0.5",
        "complete marker": success.get("status") == "complete",
        "marker seed": int(success.get("random_seed", -1)) == int(seed),
        "marker fold": int(success.get("fold", -1)) == int(fold),
    }
    failed = [name for name, passed in checks.items() if not passed]
    if failed:
        raise ValueError(f"Seed {seed}, fold {fold} failed identity checks: {failed}")

    if success.get("test_predictions_sha256") != sha256(paths["predictions"]):
        raise ValueError(f"Seed {seed}, fold {fold}: saved prediction hash mismatch.")
    if success.get("test_metrics_sha256") != sha256(paths["metrics"]):
        raise ValueError(f"Seed {seed}, fold {fold}: saved metrics hash mismatch.")

    return paths, configuration, metrics


def completed_attribution_is_valid(seed, fold, training_paths):
    root = attribution_run_root(seed, fold)
    success_path = root / "_SUCCESS.json"
    if not success_path.is_file():
        return False

    try:
        with open(success_path, "r", encoding="utf-8") as handle:
            record = json.load(handle)

        identity_ok = (
            record.get("status") == "complete"
            and record.get("experiment_name") == EXPERIMENT_NAME
            and int(record.get("seed", -1)) == int(seed)
            and int(record.get("fold", -1)) == int(fold)
            and record.get("checkpoint_sha256") == sha256(training_paths["checkpoint"])
            and record.get("input_sha256") == sha256(training_paths["input"])
            and record.get("source_test_predictions_sha256") == sha256(training_paths["predictions"])
        )
        if not identity_ok:
            return False

        for key, filename in RUN_TABLE_FILENAMES.items():
            path = root / filename
            expected_hash = record.get("table_sha256", {}).get(key)
            if not path.is_file() or expected_hash != sha256(path):
                return False
        return True
    except Exception:
        return False


def load_completed_attribution(seed, fold):
    root = attribution_run_root(seed, fold)
    return {
        key: pd.read_csv(root / filename)
        for key, filename in RUN_TABLE_FILENAMES.items()
    }

## 1.6. Exact participant-level attribution for one seed/fold checkpoint

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


def run_attribution_for_seed_fold(seed, fold):
    """Run the exact earlier attribution analysis for one held-out fold."""
    participant_rows = []
    modality_rows = []
    pairwise_rows = []
    fusion_stage_rows = []
    leave_one_out_rows = []

    print("\n" + "=" * 72)
    print(f"FOLD {fold}")
    print("=" * 72)

    checkpoint_path = checkpoint_path_for_seed_fold(seed, fold)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint was not found: {checkpoint_path}"
        )

    test_table, test_loader = build_test_loader(fold)

    model = ADNIEvidential3MTTMCModel(
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ).to(DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    three_mt_scale = float(
        F.softplus(
            model.hybrid_fusion.three_mt_scale_parameter
        ).detach().cpu().item()
    )

    tmc_scale = float(
        F.softplus(
            model.hybrid_fusion.tmc_scale_parameter
        ).detach().cpu().item()
    )

    with torch.no_grad():
        for batch in tqdm(
            test_loader,
            desc=f"Fold {fold} attribution",
            unit="batch",
        ):
            batch = move_nested_to_device(
                batch,
                DEVICE,
            )

            output = model(
                modalities=batch["modalities"],
                original_branch_masks=batch["branch_masks"],
            )

            batch_size = int(batch["target"].shape[0])

            for participant_index in range(batch_size):
                rid = int(batch["rid"][participant_index].item())
                target = int(batch["target"][participant_index].item())

                availability_vector = (
                    output["original_branch_masks"][
                        participant_index
                    ]
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(int)
                )

                availability = {
                    modality_name: int(
                        availability_vector[index]
                    )
                    for index, modality_name
                    in enumerate(MODALITY_ORDER)
                }

                modality_alpha = {
                    modality_name:
                        output["modality_opinions"][
                            modality_name
                        ]["alpha"][
                            participant_index
                        ].detach().cpu()

                    for modality_name in MODALITY_ORDER
                }

                modality_uncertainties = {}
                modality_probabilities = {}

                for modality_name in MODALITY_ORDER:
                    opinion = opinion_from_alpha(
                        modality_alpha[modality_name].unsqueeze(0)
                    )

                    available = availability[modality_name]

                    modality_uncertainty = float(
                        opinion["uncertainty"].item()
                    )

                    modality_probability = float(
                        opinion["probabilities"][0, 1].item()
                    )

                    modality_uncertainties[
                        modality_name
                    ] = modality_uncertainty

                    modality_probabilities[
                        modality_name
                    ] = modality_probability

                    modality_rows.append(
                        {
                            "FOLD": fold,
                            "RID": rid,
                            "TARGET": target,
                            "MODALITY": modality_name,
                            "MODALITY_DISPLAY":
                                MODALITY_DISPLAY_NAMES[
                                    modality_name
                                ],
                            "AVAILABLE": available,
                            "EVIDENCE_sMCI": float(
                                opinion["evidence"][0, 0].item()
                            ),
                            "EVIDENCE_pMCI": float(
                                opinion["evidence"][0, 1].item()
                            ),
                            "ALPHA_sMCI": float(
                                opinion["alpha"][0, 0].item()
                            ),
                            "ALPHA_pMCI": float(
                                opinion["alpha"][0, 1].item()
                            ),
                            "BELIEF_sMCI": float(
                                opinion["belief"][0, 0].item()
                            ),
                            "BELIEF_pMCI": float(
                                opinion["belief"][0, 1].item()
                            ),
                            "P_sMCI": float(
                                opinion["probabilities"][0, 0].item()
                            ),
                            "P_pMCI": modality_probability,
                            "STRENGTH": float(
                                opinion["strength"].item()
                            ),
                            "MODALITY_UNCERTAINTY":
                                modality_uncertainty,
                        }
                    )

                available_names = [
                    name
                    for name in MODALITY_ORDER
                    if availability[name] == 1
                ]

                available_uncertainties = {
                    name: modality_uncertainties[name]
                    for name in available_names
                }

                weakest_modality = max(
                    available_uncertainties,
                    key=available_uncertainties.get,
                )

                # Pairwise conflicts.
                participant_pairwise = []

                for first_index, first_name in enumerate(
                    available_names
                ):
                    for second_name in available_names[
                        first_index + 1:
                    ]:
                        conflict_value = pairwise_conflict(
                            modality_alpha[first_name],
                            modality_alpha[second_name],
                        )

                        probability_gap = abs(
                            modality_probabilities[first_name]
                            - modality_probabilities[second_name]
                        )

                        row = {
                            "FOLD": fold,
                            "RID": rid,
                            "TARGET": target,
                            "MODALITY_A": first_name,
                            "MODALITY_B": second_name,
                            "PAIR":
                                f"{MODALITY_DISPLAY_NAMES[first_name]}"
                                " vs "
                                f"{MODALITY_DISPLAY_NAMES[second_name]}",
                            "PAIRWISE_CONFLICT": conflict_value,
                            "ABS_PROBABILITY_GAP": probability_gap,
                        }

                        pairwise_rows.append(row)
                        participant_pairwise.append(row)

                if participant_pairwise:
                    strongest_pair = max(
                        participant_pairwise,
                        key=lambda row:
                            row["PAIRWISE_CONFLICT"],
                    )

                    max_pair_conflict = float(
                        strongest_pair[
                            "PAIRWISE_CONFLICT"
                        ]
                    )

                    strongest_pair_name = strongest_pair["PAIR"]
                else:
                    max_pair_conflict = 0.0
                    strongest_pair_name = "Not available"

                # Reconstruct full modality-specific evidence fusion from independent opinions.
                reconstructed_tmc, stage_history, _ = (
                    fuse_available_opinions(
                        modality_alpha=modality_alpha,
                        availability=availability,
                        modality_order=MODALITY_ORDER,
                    )
                )

                for stage_index, stage in enumerate(
                    stage_history,
                    start=1,
                ):
                    fusion_stage_rows.append(
                        {
                            "FOLD": fold,
                            "RID": rid,
                            "TARGET": target,
                            "STAGE_INDEX": stage_index,
                            **stage,
                        }
                    )

                original_tmc_probability = float(
                    output["tmc_output"][
                        "probabilities"
                    ][participant_index, 1].item()
                )

                original_tmc_uncertainty = float(
                    output["tmc_output"][
                        "uncertainty"
                    ][participant_index].item()
                )

                original_hybrid_probability = float(
                    output["final_output"][
                        "probabilities"
                    ][participant_index, 1].item()
                )

                original_hybrid_uncertainty = float(
                    output["final_output"][
                        "uncertainty"
                    ][participant_index].item()
                )

                three_mt_probability = float(
                    output["joint_opinion"][
                        "probabilities"
                    ][participant_index, 1].item()
                )

                three_mt_uncertainty = float(
                    output["joint_opinion"][
                        "uncertainty"
                    ][participant_index].item()
                )

                joint_evidence = (
                    output["joint_opinion"][
                        "evidence"
                    ][participant_index]
                    .detach()
                    .cpu()
                )

                participant_loo_rows = []

                if len(available_names) >= 2:
                    for removed_modality in available_names:
                        reduced_availability = dict(availability)
                        reduced_availability[
                            removed_modality
                        ] = 0

                        reduced_tmc, _, remaining_names = (
                            fuse_available_opinions(
                                modality_alpha=modality_alpha,
                                availability=reduced_availability,
                                modality_order=MODALITY_ORDER,
                            )
                        )

                        reduced_tmc_probability = float(
                            reduced_tmc[
                                "probabilities"
                            ][0, 1].item()
                        )

                        reduced_tmc_uncertainty = float(
                            reduced_tmc[
                                "uncertainty"
                            ].item()
                        )

                        reduced_hybrid = (
                            recalculate_hybrid_from_tmc(
                                joint_evidence=joint_evidence,
                                tmc_evidence=
                                    reduced_tmc[
                                        "evidence"
                                    ][0],
                                three_mt_scale=
                                    three_mt_scale,
                                tmc_scale=
                                    tmc_scale,
                            )
                        )

                        loo_row = {
                            "FOLD": fold,
                            "RID": rid,
                            "TARGET": target,
                            "REMOVED_MODALITY":
                                removed_modality,
                            "REMOVED_MODALITY_DISPLAY":
                                MODALITY_DISPLAY_NAMES[
                                    removed_modality
                                ],
                            "REMAINING_MODALITY_COUNT":
                                len(remaining_names),
                            "ORIGINAL_TMC_P_pMCI":
                                original_tmc_probability,
                            "LOO_TMC_P_pMCI":
                                reduced_tmc_probability,
                            "TMC_PROBABILITY_CHANGE":
                                reduced_tmc_probability
                                - original_tmc_probability,
                            "ABS_TMC_PROBABILITY_CHANGE":
                                abs(
                                    reduced_tmc_probability
                                    - original_tmc_probability
                                ),
                            "ORIGINAL_TMC_UNCERTAINTY":
                                original_tmc_uncertainty,
                            "LOO_TMC_UNCERTAINTY":
                                reduced_tmc_uncertainty,
                            "TMC_UNCERTAINTY_CHANGE":
                                reduced_tmc_uncertainty
                                - original_tmc_uncertainty,
                            "TMC_UNCERTAINTY_REDUCTION":
                                original_tmc_uncertainty
                                - reduced_tmc_uncertainty,
                            "ORIGINAL_HYBRID_P_pMCI":
                                original_hybrid_probability,
                            "LOO_HYBRID_P_pMCI":
                                reduced_hybrid["p_pMCI"],
                            "HYBRID_PROBABILITY_CHANGE":
                                reduced_hybrid["p_pMCI"]
                                - original_hybrid_probability,
                            "ABS_HYBRID_PROBABILITY_CHANGE":
                                abs(
                                    reduced_hybrid["p_pMCI"]
                                    - original_hybrid_probability
                                ),
                            "ORIGINAL_HYBRID_UNCERTAINTY":
                                original_hybrid_uncertainty,
                            "LOO_HYBRID_UNCERTAINTY":
                                reduced_hybrid["uncertainty"],
                            "HYBRID_UNCERTAINTY_CHANGE":
                                reduced_hybrid["uncertainty"]
                                - original_hybrid_uncertainty,
                            "HYBRID_UNCERTAINTY_REDUCTION":
                                original_hybrid_uncertainty
                                - reduced_hybrid["uncertainty"],
                            "ORIGINAL_HYBRID_CLASS":
                                int(
                                    original_hybrid_probability
                                    >= CLASSIFICATION_THRESHOLD
                                ),
                            "LOO_HYBRID_CLASS":
                                int(
                                    reduced_hybrid["p_pMCI"]
                                    >= CLASSIFICATION_THRESHOLD
                                ),
                        }

                        loo_row[
                            "HYBRID_CLASS_CHANGED"
                        ] = int(
                            loo_row[
                                "ORIGINAL_HYBRID_CLASS"
                            ]
                            !=
                            loo_row[
                                "LOO_HYBRID_CLASS"
                            ]
                        )

                        leave_one_out_rows.append(loo_row)
                        participant_loo_rows.append(loo_row)

                if participant_loo_rows:
                    strongest_reducer = max(
                        participant_loo_rows,
                        key=lambda row:
                            row[
                                "TMC_UNCERTAINTY_REDUCTION"
                            ],
                    )

                    strongest_increaser = min(
                        participant_loo_rows,
                        key=lambda row:
                            row[
                                "TMC_UNCERTAINTY_REDUCTION"
                            ],
                    )

                    strongest_uncertainty_reduction = float(
                        strongest_reducer[
                            "TMC_UNCERTAINTY_REDUCTION"
                        ]
                    )

                    uncertainty_increasing_modality = (
                        strongest_reducer[
                            "REMOVED_MODALITY_DISPLAY"
                        ]
                    )

                    uncertainty_reducing_modality = (
                        strongest_increaser[
                            "REMOVED_MODALITY_DISPLAY"
                        ]
                    )
                else:
                    strongest_uncertainty_reduction = 0.0
                    uncertainty_increasing_modality = (
                        "Not estimable"
                    )
                    uncertainty_reducing_modality = (
                        "Not estimable"
                    )

                source_category = classify_uncertainty_source(
                    available_count=len(available_names),
                    max_modality_uncertainty=float(
                        available_uncertainties[
                            weakest_modality
                        ]
                    ),
                    max_pairwise_conflict=max_pair_conflict,
                    strongest_uncertainty_reduction=
                        strongest_uncertainty_reduction,
                )

                participant_rows.append(
                    {
                        "FOLD": fold,
                        "RID": rid,
                        "TARGET": target,
                        "HYBRID_P_pMCI":
                            original_hybrid_probability,
                        "HYBRID_PREDICTED_CLASS":
                            int(
                                original_hybrid_probability
                                >= CLASSIFICATION_THRESHOLD
                            ),
                        "HYBRID_CORRECT":
                            int(
                                (
                                    original_hybrid_probability
                                    >= CLASSIFICATION_THRESHOLD
                                )
                                == target
                            ),
                        "HYBRID_UNCERTAINTY":
                            original_hybrid_uncertainty,
                        "THREE_MT_P_pMCI":
                            three_mt_probability,
                        "THREE_MT_UNCERTAINTY":
                            three_mt_uncertainty,
                        "TMC_P_pMCI":
                            original_tmc_probability,
                        "TMC_UNCERTAINTY":
                            original_tmc_uncertainty,
                        "PATHWAY_PROBABILITY_GAP":
                            abs(
                                three_mt_probability
                                - original_tmc_probability
                            ),
                        "AVAILABLE_MODALITY_COUNT":
                            len(available_names),
                        "MISSING_MODALITY_COUNT":
                            len(MODALITY_ORDER)
                            - len(available_names),
                        "MOST_UNCERTAIN_MODALITY":
                            MODALITY_DISPLAY_NAMES[
                                weakest_modality
                            ],
                        "MAX_MODALITY_UNCERTAINTY":
                            float(
                                available_uncertainties[
                                    weakest_modality
                                ]
                            ),
                        "MOST_CONFLICTING_PAIR":
                            strongest_pair_name,
                        "MAX_PAIRWISE_CONFLICT":
                            max_pair_conflict,
                        "UNCERTAINTY_INCREASING_MODALITY":
                            uncertainty_increasing_modality,
                        "UNCERTAINTY_REDUCING_MODALITY":
                            uncertainty_reducing_modality,
                        "MAX_TMC_UNCERTAINTY_REDUCTION":
                            strongest_uncertainty_reduction,
                        "DESCRIPTIVE_UNCERTAINTY_SOURCE":
                            source_category,
                        "THREE_MT_EVIDENCE_SCALE":
                            three_mt_scale,
                        "TMC_EVIDENCE_SCALE":
                            tmc_scale,
                    }
                )

    del model
    del checkpoint
    del test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()



    tables = {
        "participant": pd.DataFrame(participant_rows),
        "modality": pd.DataFrame(modality_rows),
        "pairwise": pd.DataFrame(pairwise_rows),
        "fusion": pd.DataFrame(fusion_stage_rows),
        "loo": pd.DataFrame(leave_one_out_rows),
    }

    expected_rows = int(len(test_table))
    if len(tables["participant"]) != expected_rows:
        raise ValueError(
            f"Seed {seed}, fold {fold}: expected {expected_rows} participant rows, "
            f"found {len(tables['participant'])}."
        )
    if tables["participant"]["RID"].duplicated().any():
        raise ValueError(f"Seed {seed}, fold {fold}: duplicate participant attribution rows.")

    for table in tables.values():
        table.insert(0, "SEED", int(seed))

    return tables

## 1.7. Run or safely resume all 15 checkpoint analyses

In [ ]:
def verify_prediction_reproduction(tables, seed, fold, source_prediction_path):
    observed = tables["participant"].copy()
    saved = pd.read_csv(source_prediction_path)

    required_saved = {
        "RID", "TARGET", "FINAL_P_pMCI", "FINAL_UNCERTAINTY",
        "THREE_MT_P_pMCI", "THREE_MT_UNCERTAINTY",
        "TMC_P_pMCI", "TMC_UNCERTAINTY",
    }
    missing = required_saved.difference(saved.columns)
    if missing:
        raise KeyError(
            f"Seed {seed}, fold {fold}: saved predictions lack {sorted(missing)}."
        )

    merged = observed.merge(
        saved[list(required_saved)],
        on=["RID", "TARGET"],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
    if not (merged["_merge"] == "both").all():
        raise ValueError(f"Seed {seed}, fold {fold}: RID/target reproduction mismatch.")

    comparisons = {
        "Hybrid probability": ("HYBRID_P_pMCI", "FINAL_P_pMCI"),
        "Hybrid uncertainty": ("HYBRID_UNCERTAINTY", "FINAL_UNCERTAINTY"),
        "3MT probability": ("THREE_MT_P_pMCI_x", "THREE_MT_P_pMCI_y"),
        "3MT uncertainty": ("THREE_MT_UNCERTAINTY_x", "THREE_MT_UNCERTAINTY_y"),
        "TMC probability": ("TMC_P_pMCI_x", "TMC_P_pMCI_y"),
        "TMC uncertainty": ("TMC_UNCERTAINTY_x", "TMC_UNCERTAINTY_y"),
    }

    maximum_differences = {}
    for label, (observed_column, saved_column) in comparisons.items():
        difference = np.abs(
            merged[observed_column].to_numpy(dtype=float)
            - merged[saved_column].to_numpy(dtype=float)
        )
        maximum_differences[label] = float(difference.max())
        if maximum_differences[label] > REPRODUCTION_ATOL:
            raise ValueError(
                f"Seed {seed}, fold {fold}: {label} was not reproduced. "
                f"Maximum absolute difference={maximum_differences[label]:.8g}."
            )

    return maximum_differences


def save_completed_attribution(seed, fold, tables, training_paths, reproduction):
    root = attribution_run_root(seed, fold)
    root.mkdir(parents=True, exist_ok=True)

    for key, filename in RUN_TABLE_FILENAMES.items():
        atomic_write_csv(tables[key], root / filename)

    table_hashes = {
        key: sha256(root / filename)
        for key, filename in RUN_TABLE_FILENAMES.items()
    }

    completion = {
        "status": "complete",
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_name": EXPERIMENT_NAME,
        "analysis": "fixed_fusion_repeated_seed_uncertainty_attribution",
        "seed": int(seed),
        "fold": int(fold),
        "checkpoint_sha256": sha256(training_paths["checkpoint"]),
        "input_sha256": sha256(training_paths["input"]),
        "source_test_predictions_sha256": sha256(training_paths["predictions"]),
        "prediction_reproduction_atol": float(REPRODUCTION_ATOL),
        "prediction_reproduction_maximum_absolute_differences": reproduction,
        "row_counts": {key: int(len(table)) for key, table in tables.items()},
        "table_sha256": table_hashes,
    }
    atomic_write_json(completion, root / "_SUCCESS.json")


run_inventory_rows = []

for seed in SEEDS:
    for fold in FOLDS:
        print("\n" + "=" * 88)
        print(f"SEED {seed} | FOLD {fold}")
        print("=" * 88)

        training_paths, configuration, metrics = validate_training_artifacts(seed, fold)

        if completed_attribution_is_valid(seed, fold, training_paths):
            print("A verified completed attribution exists. Skipping this run.")
            tables = load_completed_attribution(seed, fold)
            status = "reused"
            with open(attribution_run_root(seed, fold) / "_SUCCESS.json", "r") as handle:
                reproduction = json.load(handle)[
                    "prediction_reproduction_maximum_absolute_differences"
                ]
        else:
            print("Running inference and attribution from the saved best checkpoint.")
            tables = run_attribution_for_seed_fold(seed, fold)
            reproduction = verify_prediction_reproduction(
                tables=tables,
                seed=seed,
                fold=fold,
                source_prediction_path=training_paths["predictions"],
            )
            save_completed_attribution(
                seed=seed,
                fold=fold,
                tables=tables,
                training_paths=training_paths,
                reproduction=reproduction,
            )
            status = "computed"

        run_inventory_rows.append(
            {
                "SEED": int(seed),
                "FOLD": int(fold),
                "STATUS": status,
                "PARTICIPANTS": int(len(tables["participant"])),
                "CHECKPOINT_EPOCH": int(metrics["checkpoint_epoch"]),
                "CHECKPOINT_VALIDATION_AUC": float(metrics["checkpoint_validation_auc"]),
                "MAX_REPRODUCTION_DIFFERENCE": float(max(reproduction.values())),
                "RUN_OUTPUT_ROOT": str(attribution_run_root(seed, fold)),
            }
        )

        del tables
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


run_inventory = pd.DataFrame(run_inventory_rows)
atomic_write_csv(run_inventory, TABLE_DIR / "validated_attribution_run_inventory.csv")
display(run_inventory)
print("\nAll 15 seed/fold attributions are present and prediction-reproducing.")

## 1.8. Consolidate three complete 544-participant out-of-fold seed evaluations

In [ ]:
all_tables = {key: [] for key in RUN_TABLE_FILENAMES}

for seed in SEEDS:
    for fold in FOLDS:
        training_paths, _, _ = validate_training_artifacts(seed, fold)
        if not completed_attribution_is_valid(seed, fold, training_paths):
            raise RuntimeError(f"Seed {seed}, fold {fold} is not safely complete.")
        run_tables = load_completed_attribution(seed, fold)
        for key, table in run_tables.items():
            all_tables[key].append(table)

all_tables = {
    key: pd.concat(tables, ignore_index=True)
    for key, tables in all_tables.items()
}

participant_attribution = all_tables["participant"]
modality_attribution = all_tables["modality"]
pairwise_conflict_table = all_tables["pairwise"]
fusion_history_table = all_tables["fusion"]
leave_one_out_table = all_tables["loo"]

seed_counts = participant_attribution.groupby("SEED").agg(
    ROWS=("RID", "size"),
    UNIQUE_RIDS=("RID", "nunique"),
)
if not (seed_counts["ROWS"] == EXPECTED_PARTICIPANTS_PER_SEED).all():
    raise ValueError("At least one seed does not have 544 OOF participant rows.")
if not (seed_counts["UNIQUE_RIDS"] == EXPECTED_PARTICIPANTS_PER_SEED).all():
    raise ValueError("At least one seed has duplicate or missing OOF RIDs.")

reference_identity = (
    participant_attribution.loc[
        participant_attribution["SEED"] == 42,
        ["FOLD", "RID", "TARGET"],
    ]
    .sort_values(["FOLD", "RID"])
    .reset_index(drop=True)
)
for seed in (17, 73):
    candidate = (
        participant_attribution.loc[
            participant_attribution["SEED"] == seed,
            ["FOLD", "RID", "TARGET"],
        ]
        .sort_values(["FOLD", "RID"])
        .reset_index(drop=True)
    )
    if not candidate.equals(reference_identity):
        raise ValueError(f"Seed {seed} has a different OOF participant identity.")

for key, filename in RUN_TABLE_FILENAMES.items():
    atomic_write_csv(all_tables[key], TABLE_DIR / f"all_seeds_{filename}")

print("Participant counts by complete seed run:")
display(seed_counts)

## 1.9. Quantify attribution stability across seeds

In [ ]:
# Each seed contributes one complete 544-participant OOF evaluation.
# The across-seed SD below therefore uses N=3, not N=15.

available_modality = modality_attribution.loc[
    modality_attribution["AVAILABLE"] == 1
].copy()

seedwise_evidence = (
    available_modality
    .groupby(["SEED", "MODALITY", "MODALITY_DISPLAY"])
    .agg(
        AVAILABLE_PARTICIPANTS=("RID", "count"),
        MEAN_MODALITY_UNCERTAINTY=("MODALITY_UNCERTAINTY", "mean"),
        MEDIAN_MODALITY_UNCERTAINTY=("MODALITY_UNCERTAINTY", "median"),
        MEAN_STRENGTH=("STRENGTH", "mean"),
    )
    .reset_index()
)

seedwise_loo = (
    leave_one_out_table
    .groupby(["SEED", "REMOVED_MODALITY", "REMOVED_MODALITY_DISPLAY"])
    .agg(
        PARTICIPANTS=("RID", "count"),
        MEAN_ABS_TMC_PROBABILITY_CHANGE=("ABS_TMC_PROBABILITY_CHANGE", "mean"),
        MEAN_ABS_HYBRID_PROBABILITY_CHANGE=("ABS_HYBRID_PROBABILITY_CHANGE", "mean"),
        MEAN_TMC_UNCERTAINTY_REDUCTION=("TMC_UNCERTAINTY_REDUCTION", "mean"),
        MEAN_HYBRID_UNCERTAINTY_REDUCTION=("HYBRID_UNCERTAINTY_REDUCTION", "mean"),
        HYBRID_CLASS_CHANGES=("HYBRID_CLASS_CHANGED", "sum"),
    )
    .reset_index()
)

seedwise_conflict = (
    pairwise_conflict_table
    .groupby(["SEED", "PAIR"])
    .agg(
        PARTICIPANT_PAIRS=("RID", "count"),
        MEAN_PAIRWISE_CONFLICT=("PAIRWISE_CONFLICT", "mean"),
        MEAN_ABS_PROBABILITY_GAP=("ABS_PROBABILITY_GAP", "mean"),
    )
    .reset_index()
)

seedwise_evidence["UNCERTAINTY_RANK"] = (
    seedwise_evidence.groupby("SEED")["MEAN_MODALITY_UNCERTAINTY"]
    .rank(method="min", ascending=False)
    .astype(int)
)
seedwise_loo["INFLUENCE_RANK"] = (
    seedwise_loo.groupby("SEED")["MEAN_ABS_HYBRID_PROBABILITY_CHANGE"]
    .rank(method="min", ascending=False)
    .astype(int)
)
seedwise_loo["CLASS_CHANGE_RANK"] = (
    seedwise_loo.groupby("SEED")["HYBRID_CLASS_CHANGES"]
    .rank(method="min", ascending=False)
    .astype(int)
)

evidence_across_seeds = (
    seedwise_evidence
    .groupby(["MODALITY", "MODALITY_DISPLAY"])
    .agg(
        N_SEEDS=("SEED", "count"),
        MEAN_MODALITY_UNCERTAINTY=("MEAN_MODALITY_UNCERTAINTY", "mean"),
        SD_MODALITY_UNCERTAINTY=("MEAN_MODALITY_UNCERTAINTY", "std"),
        MEAN_STRENGTH=("MEAN_STRENGTH", "mean"),
        SD_STRENGTH=("MEAN_STRENGTH", "std"),
        MEAN_UNCERTAINTY_RANK=("UNCERTAINTY_RANK", "mean"),
        MIN_UNCERTAINTY_RANK=("UNCERTAINTY_RANK", "min"),
        MAX_UNCERTAINTY_RANK=("UNCERTAINTY_RANK", "max"),
    )
    .reset_index()
    .sort_values("MEAN_MODALITY_UNCERTAINTY", ascending=False)
)

loo_across_seeds = (
    seedwise_loo
    .groupby(["REMOVED_MODALITY", "REMOVED_MODALITY_DISPLAY"])
    .agg(
        N_SEEDS=("SEED", "count"),
        MEAN_ABS_HYBRID_PROBABILITY_CHANGE=("MEAN_ABS_HYBRID_PROBABILITY_CHANGE", "mean"),
        SD_ABS_HYBRID_PROBABILITY_CHANGE=("MEAN_ABS_HYBRID_PROBABILITY_CHANGE", "std"),
        MEAN_HYBRID_CLASS_CHANGES=("HYBRID_CLASS_CHANGES", "mean"),
        SD_HYBRID_CLASS_CHANGES=("HYBRID_CLASS_CHANGES", "std"),
        MEAN_HYBRID_UNCERTAINTY_REDUCTION=("MEAN_HYBRID_UNCERTAINTY_REDUCTION", "mean"),
        SD_HYBRID_UNCERTAINTY_REDUCTION=("MEAN_HYBRID_UNCERTAINTY_REDUCTION", "std"),
        MEAN_INFLUENCE_RANK=("INFLUENCE_RANK", "mean"),
        MIN_INFLUENCE_RANK=("INFLUENCE_RANK", "min"),
        MAX_INFLUENCE_RANK=("INFLUENCE_RANK", "max"),
    )
    .reset_index()
    .sort_values("MEAN_ABS_HYBRID_PROBABILITY_CHANGE", ascending=False)
)

conflict_across_seeds = (
    seedwise_conflict
    .groupby("PAIR")
    .agg(
        N_SEEDS=("SEED", "count"),
        MEAN_PAIRWISE_CONFLICT=("MEAN_PAIRWISE_CONFLICT", "mean"),
        SD_PAIRWISE_CONFLICT=("MEAN_PAIRWISE_CONFLICT", "std"),
        MEAN_ABS_PROBABILITY_GAP=("MEAN_ABS_PROBABILITY_GAP", "mean"),
        SD_ABS_PROBABILITY_GAP=("MEAN_ABS_PROBABILITY_GAP", "std"),
    )
    .reset_index()
    .sort_values("MEAN_PAIRWISE_CONFLICT", ascending=False)
)

participant_seed_average = (
    participant_attribution
    .groupby(["FOLD", "RID", "TARGET"], as_index=False)
    .agg(
        SEEDS_OBSERVED=("SEED", "nunique"),
        MEAN_HYBRID_P_pMCI=("HYBRID_P_pMCI", "mean"),
        SD_HYBRID_P_pMCI=("HYBRID_P_pMCI", "std"),
        MEAN_HYBRID_UNCERTAINTY=("HYBRID_UNCERTAINTY", "mean"),
        MEAN_MAX_MODALITY_UNCERTAINTY=("MAX_MODALITY_UNCERTAINTY", "mean"),
        MEAN_MAX_PAIRWISE_CONFLICT=("MAX_PAIRWISE_CONFLICT", "mean"),
        MEAN_PATHWAY_PROBABILITY_GAP=("PATHWAY_PROBABILITY_GAP", "mean"),
    )
)
if not (participant_seed_average["SEEDS_OBSERVED"] == len(SEEDS)).all():
    raise ValueError("At least one participant is missing a seed-level attribution.")

summary_tables = {
    "seedwise_available_modality_evidence.csv": seedwise_evidence,
    "seedwise_leave_one_modality_out.csv": seedwise_loo,
    "seedwise_pairwise_conflict.csv": seedwise_conflict,
    "across_seed_available_modality_evidence_summary.csv": evidence_across_seeds,
    "across_seed_leave_one_out_summary.csv": loo_across_seeds,
    "across_seed_pairwise_conflict_summary.csv": conflict_across_seeds,
    "participant_seed_averaged_attribution.csv": participant_seed_average,
}
for filename, table in summary_tables.items():
    atomic_write_csv(table, TABLE_DIR / filename)

print("Across-seed available-opinion uncertainty (three seed replicates):")
display(evidence_across_seeds.round(6))
print("\nAcross-seed leave-one-opinion-out influence (three seed replicates):")
display(loo_across_seeds.round(6))
print("\nHighest-conflict pairs across seeds:")
display(conflict_across_seeds.head(15).round(6))

print("\nInfluence rank by seed:")
display(
    seedwise_loo.pivot(
        index="REMOVED_MODALITY_DISPLAY",
        columns="SEED",
        values="INFLUENCE_RANK",
    ).sort_values(list(SEEDS))
)

## 1.10. Dissertation-ready across-seed figures

In [ ]:
TEAL = "#2B7A78"
PALE_TEAL = "#A7DADC"
GREY_BLUE = "#577590"


def seed_point_positions(base_position, count):
    offsets = np.linspace(-0.12, 0.12, count)
    return base_position + offsets


# Cross-seed influence: mean absolute Hybrid probability change.
plot_table = loo_across_seeds.sort_values(
    "MEAN_ABS_HYBRID_PROBABILITY_CHANGE", ascending=True
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
y = np.arange(len(plot_table))
ax.barh(
    y,
    plot_table["MEAN_ABS_HYBRID_PROBABILITY_CHANGE"],
    xerr=plot_table["SD_ABS_HYBRID_PROBABILITY_CHANGE"],
    color=PALE_TEAL,
    edgecolor=TEAL,
    capsize=4,
    label="Mean ± SD across seeds",
)
for position, modality in zip(y, plot_table["REMOVED_MODALITY_DISPLAY"]):
    seed_values = seedwise_loo.loc[
        seedwise_loo["REMOVED_MODALITY_DISPLAY"] == modality,
        "MEAN_ABS_HYBRID_PROBABILITY_CHANGE",
    ].to_numpy()
    ax.scatter(
        seed_values,
        seed_point_positions(position, len(seed_values)),
        color=GREY_BLUE,
        s=28,
        zorder=3,
    )
ax.set_yticks(y, plot_table["REMOVED_MODALITY_DISPLAY"])
ax.set_xlabel("Mean absolute Hybrid probability change after opinion removal")
ax.set_ylabel("Removed modality opinion")
ax.set_title("Fixed-fusion modality influence across three complete seed runs")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "across_seed_leave_one_out_hybrid_probability_change.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


# Cross-seed uncertainty of each available modality opinion.
plot_table = evidence_across_seeds.sort_values(
    "MEAN_MODALITY_UNCERTAINTY", ascending=True
).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
y = np.arange(len(plot_table))
ax.barh(
    y,
    plot_table["MEAN_MODALITY_UNCERTAINTY"],
    xerr=plot_table["SD_MODALITY_UNCERTAINTY"],
    color="#C9D6E4",
    edgecolor=GREY_BLUE,
    capsize=4,
    label="Mean ± SD across seeds",
)
for position, modality in zip(y, plot_table["MODALITY_DISPLAY"]):
    seed_values = seedwise_evidence.loc[
        seedwise_evidence["MODALITY_DISPLAY"] == modality,
        "MEAN_MODALITY_UNCERTAINTY",
    ].to_numpy()
    ax.scatter(
        seed_values,
        seed_point_positions(position, len(seed_values)),
        color=TEAL,
        s=28,
        zorder=3,
    )
ax.set_yticks(y, plot_table["MODALITY_DISPLAY"])
ax.set_xlabel("Mean uncertainty of available modality opinions")
ax.set_ylabel("Modality")
ax.set_title("Available-opinion uncertainty across three complete seed runs")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "across_seed_available_modality_uncertainty.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 1.11. Final integrity checks and completion manifest

In [ ]:
top_influence_by_seed = (
    seedwise_loo.sort_values(["SEED", "INFLUENCE_RANK"])
    .groupby("SEED", as_index=False)
    .first()[
        [
            "SEED",
            "REMOVED_MODALITY_DISPLAY",
            "MEAN_ABS_HYBRID_PROBABILITY_CHANGE",
            "HYBRID_CLASS_CHANGES",
        ]
    ]
)

top_uncertainty_by_seed = (
    seedwise_evidence.sort_values(["SEED", "UNCERTAINTY_RANK"])
    .groupby("SEED", as_index=False)
    .first()[
        ["SEED", "MODALITY_DISPLAY", "MEAN_MODALITY_UNCERTAINTY"]
    ]
)

manifest = {
    "status": "complete",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_name": EXPERIMENT_NAME,
    "analysis": "fixed_fusion_repeated_seed_uncertainty_attribution",
    "seeds": list(SEEDS),
    "folds": list(FOLDS),
    "independent_seed_replicates": len(SEEDS),
    "validated_checkpoint_count": len(SEEDS) * len(FOLDS),
    "participants_per_seed": EXPECTED_PARTICIPANTS_PER_SEED,
    "all_seed_participant_rows": int(len(participant_attribution)),
    "maximum_prediction_reproduction_difference": float(
        run_inventory["MAX_REPRODUCTION_DIFFERENCE"].max()
    ),
    "top_influence_by_seed": top_influence_by_seed.to_dict(orient="records"),
    "top_available_opinion_uncertainty_by_seed": (
        top_uncertainty_by_seed.to_dict(orient="records")
    ),
    "table_sha256": {
        path.name: sha256(path)
        for path in sorted(TABLE_DIR.glob("*.csv"))
    },
}
atomic_write_json(manifest, OUTPUT_ROOT / "_SUCCESS.json")

print("=" * 88)
print("REPEATED-SEED UNCERTAINTY ATTRIBUTION COMPLETE")
print("=" * 88)
print(f"Validated checkpoints: {len(SEEDS) * len(FOLDS)}")
print(f"Complete OOF seed evaluations: {len(SEEDS)}")
print(f"Participants per seed: {EXPECTED_PARTICIPANTS_PER_SEED}")
print(
    "Maximum difference from the original saved test predictions: "
    f"{run_inventory['MAX_REPRODUCTION_DIFFERENCE'].max():.8g}"
)
print("\nTop leave-one-opinion-out influence in each seed:")
display(top_influence_by_seed.round(6))
print("\nMost uncertain available opinion in each seed:")
display(top_uncertainty_by_seed.round(6))
print(f"\nAll outputs were saved to:\n{OUTPUT_ROOT}")
print(
    "\nFor the dissertation, report mean ± SD across the three complete seed runs. "
    "Do not describe the 15 fold checkpoints as 15 independent replicates."
)

## 1.12. Interpretation boundary

Leave-one-opinion-out analysis measures sensitivity of the trained evidential fusion calculation. It does **not** estimate the causal or clinical value of collecting a modality, because removing an opinion does not retrain the encoders and naturally available modality groups differ between participants.

When writing the dissertation, we will distinguish three questions:

1. Which available modality opinions are individually most uncertain?
2. Which opinion removals change the fixed Hybrid prediction most?
3. Are those rankings consistent across seeds?

We will write the Results section only after inspecting these outputs together.